# Этап 5: многопроцессный прогрев disk cache

## Goal

Проверить `NumpyDataset.warm_cache()` на небольшой реальной части EEG/EOG-корпуса: последовательно, повторно по готовому кэшу и с двумя worker-процессами.

## Setup

Ноутбук использует только `S_1/Trial_1` из `data/Data_Pattern` и шаг `patt`. Исходные FIF остаются неизменными; производные `.npy` записываются в `artifacts/cache/notebook-warmup-demo/`.

In [1]:
from dataclasses import asdict
from pathlib import Path
import shutil
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.datasets import CacheWarmupReport, NumpyDataset

DATASET_DIR = PROJECT_ROOT / "data" / "Data_Pattern"
CACHE_DIR = PROJECT_ROOT / "artifacts" / "cache" / "notebook-warmup-demo" / "float32"

DATASET_DIR, CACHE_DIR

(PosixPath('/home/slauva/Projects/master-thesis-2024-2026/code/data/Data_Pattern'),
 PosixPath('/home/slauva/Projects/master-thesis-2024-2026/code/artifacts/cache/notebook-warmup-demo/float32'))

### Key Assumptions

- `max_workers=1` — детерминированный последовательный режим для отладки.
- `max_workers=2` — отдельные процессы, каждый создаёт только disk cache.
- Сравнение времени здесь демонстрационное: девять блоков недостаточно для устойчивого benchmark.
- `processed` означает созданную запись, `cached` — уже валидную запись, `failed` — ошибку, `skipped` — не обработанный после `fail_fast` элемент.

## Steps

### 1. Соберём ограниченный индекс

Пустой список trial в `exclude_samples` исключает субъекта целиком. Для `S_1` исключаем только `Trial_2`.

In [2]:
subject_dirs = sorted(path for path in DATASET_DIR.glob("S_*") if path.is_dir())
exclude_samples = {path.name: [] for path in subject_dirs if path.name != "S_1"}
exclude_samples["S_1"] = ["Trial_2"]

dataset = NumpyDataset(
    DATASET_DIR,
    dataset_step_type="patt",
    cache_policy="both",
    cache_dir=CACHE_DIR,
    exclude_samples=exclude_samples,
)

sample_keys = [
    (sample.subject_id, sample.trial_number, sample.block_index)
    for sample in dataset.samples
]
pd.DataFrame(sample_keys, columns=["subject_id", "trial_number", "block_index"])

,subject_id,trial_number,block_index
0,1,1,1
1,1,1,2
2,1,1,3
3,1,1,4
4,1,1,5
5,1,1,6
6,1,1,7
7,1,1,8
8,1,1,9


### 2. Последовательный cold start

Удаляем только демонстрационный кэш и создаём девять записей заново.

In [3]:
shutil.rmtree(CACHE_DIR, ignore_errors=True)
sequential_report = dataset.warm_cache(max_workers=1)
pd.DataFrame([asdict(sequential_report)])

,processed,cached,skipped,failed,errors,max_workers,duration_seconds
0,9,0,0,0,(),1,0.156785


### 3. Повторный прогрев

Второй вызов валидирует manifest и массивы, но не открывает FIF для повторной конвертации.

In [4]:
cached_report = dataset.warm_cache(max_workers=1)
pd.DataFrame([asdict(cached_report)])

,processed,cached,skipped,failed,errors,max_workers,duration_seconds
0,0,9,0,0,(),1,0.029307


### 4. Cold start с двумя процессами

Снова очищаем только demo-каталог. Worker-процессы независимо читают FIF и атомарно создают записи disk cache.

In [5]:
shutil.rmtree(CACHE_DIR, ignore_errors=True)
parallel_report = dataset.warm_cache(max_workers=2)

timings = pd.DataFrame(
    [
        {"mode": "sequential", **asdict(sequential_report)},
        {"mode": "two workers", **asdict(parallel_report)},
        {"mode": "cache hit", **asdict(cached_report)},
    ]
)
timings

,mode,processed,cached,skipped,failed,errors,max_workers,duration_seconds
0,sequential,9,0,0,0,(),1,0.156785
1,two workers,9,0,0,0,(),2,0.107001
2,cache hit,0,9,0,0,(),1,0.029307


### 5. Проверим disk cache и RAM родителя

`warm_cache()` не должен раздувать память процесса ноутбука. Первый обычный `dataset[0]` загружает готовые `.npy` и уже после этого помещает элемент в локальный LRU.

In [6]:
manifest_paths = sorted(CACHE_DIR.glob("S_*/Trial_*/Block_*/manifest.json"))
memory_items_after_warmup = dataset.memory_cache_items
loaded = dataset[0]

cache_check = {
    "manifest_count": len(manifest_paths),
    "memory_items_after_warmup": memory_items_after_warmup,
    "memory_items_after_getitem": dataset.memory_cache_items,
    "sample_key": (
        loaded.sample.subject_id,
        loaded.sample.trial_number,
        loaded.sample.block_index,
    ),
    "eeg_shape": loaded.eeg.shape,
    "eog_shape": loaded.eog.shape,
}
pd.Series(cache_check, name="value")

manifest_count                          9
memory_items_after_warmup               0
memory_items_after_getitem              1
sample_key                      (1, 1, 1)
eeg_shape                     (63, 26001)
eog_shape                      (5, 26001)
Name: value, dtype: object

## Checks

Проверки фиксируют контракт API, а не конкретное время выполнения.

In [7]:
assert isinstance(sequential_report, CacheWarmupReport)
assert len(dataset) == 9
assert sequential_report.total == 9
assert sequential_report.processed == 9
assert sequential_report.failed == 0
assert cached_report.cached == 9
assert parallel_report.processed == 9
assert parallel_report.max_workers == 2
assert parallel_report.failed == 0
assert len(manifest_paths) == 9
assert memory_items_after_warmup == 0
assert dataset.memory_cache_items == 1

print("Все проверки этапа 5 пройдены.")

Все проверки этапа 5 пройдены.


## Next Steps

- Для полного корпуса убрать `exclude_samples` и вызвать `warm_cache()`; по умолчанию будет использовано `min(4, os.cpu_count() or 1)` процессов.
- Для диагностического прогона использовать `fail_fast=False` и разобрать `report.errors`.
- Не считать ускорение из этого ноутбука benchmark: измерять отдельно на полном наборе, одинаковом состоянии filesystem cache и нескольких повторах.
- Следующий прикладной слой может строить окна и признаки поверх `LoadedSample`, не смешивая это с индексированием и чтением FIF.